In [1]:
import os
import shutil
from datetime import datetime

# Path to the parent folder (Downloads\Balance  Secondary PR)
downloads_path = r"C:\Users\IN10052060\Downloads\CONT+PR $25 Variance_Live"

# Target folder name (can be timestamped if you want)
folder_name = "file"  # e.g., datetime.now().strftime("file_%Y%m%d_%H%M%S")
folder_path = os.path.join(downloads_path, folder_name)

# 1) If the folder exists, delete all contents inside it
if os.path.isdir(folder_path):
    for entry in os.listdir(folder_path):
        entry_path = os.path.join(folder_path, entry)
        try:
            if os.path.isfile(entry_path) or os.path.islink(entry_path):
                os.remove(entry_path)  # delete file or symlink
            elif os.path.isdir(entry_path):
                shutil.rmtree(entry_path)  # delete subfolder
        except Exception as e:
            print(f"Could not delete '{entry_path}': {e}")
else:
    # 2) If it doesn't exist, create it (including any missing parents)
    os.makedirs(folder_path, exist_ok=True)

# (Re)ensure the folder exists after cleanup
os.makedirs(folder_path, exist_ok=True)

print(f"Folder ready: {folder_path}")

Folder ready: C:\Users\IN10052060\Downloads\CONT+PR $25 Variance_Live\file


In [2]:
# -*- coding: utf-8 -*-
"""
Batch SQL Runner with per-server per-database status and summary

- Reads evaluated Server/Database from Excel (data_only=True)
- Skips rows where cells are formulas (start with '='), blanks, or None
- Tries every database listed under a server
- Logs per-database status in 'Log' sheet
- Adds 'Summary' sheet with success/failure counts per server
- Prints console summary per SQL file: total DBs succeeded vs. failed, and failed DB@Server list

Enhancements:
- Robust multi-statement batch execution using pyodbc cursor + nextset()
- No pandas.read_sql_query() for multi-batch scripts
- Auto-injects SET NOCOUNT ON; at top (if missing)
- Unescapes HTML entities (&gt;,&lt;,&amp;) in SQL text
- Removes named PRIMARY KEY constraints on temp tables (e.g., CONSTRAINT PK_#act ...)
  to avoid cross-session collisions (SQL 2714)
- Small pause between DBs to reduce tempdb overlap races
"""

import os
import sys
import re
import time
import html
import warnings
from datetime import datetime
from typing import List, Tuple, Optional

import pandas as pd
import pyodbc
from openpyxl import Workbook, load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

try:
    import winsound
except ImportError:
    winsound = None

warnings.filterwarnings("ignore", category=UserWarning)

# ----------------------------- #
# --------- CONFIGURE --------- #
# ----------------------------- #

DB_STRUCTURE_PATH = r"C:\Users\IN10052060\Desktop\Python\SqlDBStructure.xlsx"
EXCEL_SHEET_NAME = "All Sites"  # Columns: A=Facility, B=Database, C=Server


SQL_FOLDER = r"C:\Users\IN10052060\Downloads\CONT+PR $25 Variance_Live\SQL Query"
OUTPUT_DIR = r"C:\Users\IN10052060\Downloads\CONT+PR $25 Variance_Live\file"




DEFAULT_TIMEOUT_SECONDS = 120   # Allow heavier batches to finish
ENABLE_ODBC_ENCRYPTION = False  # Set True if your env requires encryption

DB_PAUSE_SECONDS = 0.20         # Small pause between DBs to reduce tempdb races

# ----------------------------- #
# ---------- HELPERS ---------- #
# ----------------------------- #

def ensure_dir(path: str) -> None:
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)

def safe_strip(val) -> Optional[str]:
    if val is None:
        return None
    if isinstance(val, str):
        s = val.strip()
        return s if s else None
    return str(val).strip()

def is_invalid_cell(text: Optional[str]) -> bool:
    if text is None:
        return True
    t = text.strip()
    if not t:
        return True
    if t.startswith("="):
        return True
    return False

def read_db_targets_from_excel(xlsx_path: str, sheet_name: str) -> pd.DataFrame:
    try:
        wb = load_workbook(xlsx_path, data_only=True)
    except Exception as e:
        raise RuntimeError(f"Failed to open DB structure file: {xlsx_path}\n{e}")

    if sheet_name not in wb.sheetnames:
        raise ValueError(f"Sheet '{sheet_name}' not found in '{xlsx_path}'. Available: {wb.sheetnames}")

    ws = wb[sheet_name]

    targets: List[Tuple[Optional[str], str, str]] = []
    bad_rows: List[Tuple[int, Optional[str], Optional[str], Optional[str]]] = []

    # A=Facility, B=Database, C=Server
    for r in range(2, ws.max_row + 1):
        facility = safe_strip(ws.cell(row=r, column=1).value)
        database = safe_strip(ws.cell(row=r, column=2).value)
        server   = safe_strip(ws.cell(row=r, column=3).value)

        if is_invalid_cell(server) or is_invalid_cell(database):
            bad_rows.append((r, facility, database, server))
            continue

        targets.append((facility, database, server))

    if bad_rows:
        print("⚠️ Skipped rows with missing/formula values (row, facility, database, server):")
        for item in bad_rows[:10]:
            print("   ", item)
        if len(bad_rows) > 10:
            print(f"   ... and {len(bad_rows) - 10} more")

    if not targets:
        raise ValueError(
            "No valid Server/Database rows found.\n"
            "- Ensure formulas are evaluated (open & save in Excel with Ctrl+Alt+F9).\n"
            "- Or replace formulas with static values."
        )

    df = pd.DataFrame(targets, columns=["Facility", "Database", "Server"])
    df.drop_duplicates(inplace=True)
    df = df.dropna(subset=["Database", "Server"])
    return df

def read_sql_file(path: str) -> str:
    encodings = ["utf-8-sig", "utf-8", "cp1252", "latin-1"]
    for enc in encodings:
        try:
            with open(path, "r", encoding=enc) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def sanitize_query(sql_text: str) -> str:
    """
    - Removes standalone GO lines
    - Unescapes HTML entities (&gt;,&lt;,&amp;)
    - Converts named PK constraints on temp tables to unnamed PRIMARY KEY
      to avoid SQL 2714 in tempdb across concurrent sessions.
    - Injects SET NOCOUNT ON; at top if missing
    """
    # Remove 'GO' batch separators
    lines = []
    for line in sql_text.splitlines():
        if line.strip().upper() == "GO":
            continue
        lines.append(line)
    text = "\n".join(lines)

    # Unescape HTML entities (&gt; -> >, etc.)
    text = html.unescape(text)

    # Replace named temp-table PK constraints like:
    #   CONSTRAINT PK_#act PRIMARY KEY ...
    # or any constraint whose name contains '#'
    # with an unnamed PRIMARY KEY
    # Handles optional [brackets] and case-insensitive
    text = re.sub(
        r"CONSTRAINT\s+\[?(?:PK_)?[^ \]]*#\w+\]?\s+PRIMARY\s+KEY",
        "PRIMARY KEY",
        text,
        flags=re.IGNORECASE
    )

    # Ensure SET NOCOUNT ON; is at top
    header = text.lstrip()
    if not header.upper().startswith("SET NOCOUNT ON;"):
        text = "SET NOCOUNT ON;\n" + text

    # (Optional) Also add XACT_ABORT for cleaner failure behavior
    header2 = text.lstrip()
    if not header2.upper().startswith("SET XACT_ABORT ON;"):
        text = "SET XACT_ABORT ON;\n" + text

    return text

def clean_illegal_characters(value):
    if isinstance(value, str):
        return "".join(ch for ch in value if ch.isprintable())
    return value

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df_cleaned = df.copy()
    for col in df_cleaned.columns:
        if df_cleaned[col].dtype == "object":
            df_cleaned[col] = df_cleaned[col].apply(clean_illegal_characters)
    df_cleaned.columns = df_cleaned.columns.str.strip()
    df_cleaned.drop_duplicates(inplace=True)
    return df_cleaned

def pick_best_odbc_driver() -> str:
    try:
        drivers = pyodbc.drivers()
    except Exception:
        drivers = []
    for cand in ("ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server", "SQL Server"):
        if cand in drivers:
            return cand
    return "SQL Server"

# ----------------------------- #
#    CORE: ROBUST FETCH DATA    #
# ----------------------------- #

def fetch_data(
    database: str,
    server: str,
    query: str,
    timeout: int = DEFAULT_TIMEOUT_SECONDS,
    encrypt: bool = ENABLE_ODBC_ENCRYPTION
) -> Tuple[Optional[pd.DataFrame], int, float, str]:
    """
    Execute an entire multi-statement batch in ONE call.
    Advance through non-result sets until the first set with columns is found.
    Build DataFrame manually—no pandas.read_sql_query (which expects a result set immediately).
    """
    start = datetime.now()
    conn = None
    try:
        driver = pick_best_odbc_driver()
        conn_str = (
            f"Driver={{{driver}}};"
            f"Server={server};"
            f"Database={database};"
            "Trusted_Connection=Yes;"
            f"Timeout={timeout};"
        )
        if encrypt and ("ODBC Driver 17" in driver or "ODBC Driver 18" in driver):
            conn_str += "Encrypt=Yes;TrustServerCertificate=Yes;"

        print(f"Connecting to DB='{database}' on Server='{server}' using Driver='{driver}' ...")
        conn = pyodbc.connect(conn_str, timeout=timeout, autocommit=True)
        cur = conn.cursor()

        # Execute entire script once (temp tables persist within this connection)
        cur.execute(query)

        # Walk through intermediate sets until we find the first result set with columns
        columns = None
        rows = None

        while True:
            if cur.description:
                columns = [c[0] for c in cur.description]
                rows = cur.fetchall()
                break
            if not cur.nextset():
                break  # No more sets; no result set produced

        duration = (datetime.now() - start).total_seconds()

        # No result set -> return empty DataFrame but Success
        if not columns:
            return (pd.DataFrame(), 0, duration, "Success")

        df = pd.DataFrame.from_records(rows, columns=columns)
        df_cleaned = clean_dataframe(df)
        return (df_cleaned, len(df_cleaned), duration, "Success")

    except pyodbc.Error as e:
        duration = (datetime.now() - start).total_seconds()
        status = f"pyodbc.Error: {e}"
        print(f"❌ {status}")
        return (None, 0, duration, status)
    except Exception as e:
        duration = (datetime.now() - start).total_seconds()
        status = f"Error: {type(e).__name__}: {e}"
        print(f"❌ {status}")
        return (None, 0, duration, status)
    finally:
        if conn is not None:
            try:
                conn.close()
            except Exception:
                pass

def beep_success():
    try:
        if winsound:
            winsound.MessageBeep()
    except Exception:
        pass

# ----------------------------- #
# ------------ MAIN ----------- #
# ----------------------------- #

def main():
    # Validate inputs
    if not os.path.exists(DB_STRUCTURE_PATH):
        print(f"❌ DB_STRUCTURE_PATH not found: {DB_STRUCTURE_PATH}")
        sys.exit(1)
    if not os.path.exists(SQL_FOLDER):
        print(f"❌ SQL_FOLDER not found: {SQL_FOLDER}")
        sys.exit(1)
    ensure_dir(OUTPUT_DIR)

    # Load targets
    print("📘 Loading DB targets from Excel (evaluated values)...")
    df_targets = read_db_targets_from_excel(DB_STRUCTURE_PATH, EXCEL_SHEET_NAME)
    if df_targets.empty:
        print("❌ No valid targets after filtering.")
        sys.exit(1)

    servers_grouped = df_targets.groupby("Server", dropna=True)

    # Collect SQL files
    sql_files = [f for f in os.listdir(SQL_FOLDER) if f.lower().endswith((".sql", ".txt"))]
    if not sql_files:
        print(f"❌ No .sql or .txt files found in: {SQL_FOLDER}")
        sys.exit(1)

    # Process each SQL file
    for sql_file in sql_files:
        sql_path = os.path.join(SQL_FOLDER, sql_file)
        print(f"\n🔍 Running query from: {sql_file}")

        raw_query = read_sql_file(sql_path)
        query = sanitize_query(raw_query)

        # --- Overall counters across all servers for this SQL file ---
        overall_total = 0
        overall_succeeded = 0
        overall_failed = 0
        overall_failed_list: List[str] = []  # e.g., "TranMTTN@AHS-TRAN15.accretivehealth.local"

        for server, group in servers_grouped:
            print(f"\n➡️ Processing Server: {server}")

            # Prepare Excel workbook for this server
            master_file = Workbook()
            log_sheet = master_file.active
            log_sheet.title = "Log"
            raw_data_sheet = master_file.create_sheet("RawData")
            summary_sheet = master_file.create_sheet("Summary")

            log_sheet.append(["Server", "Database", "Status", "Row Count", "Duration (s)", "Timestamp"])
            summary_sheet.append(["Server", "Total Databases", "Succeeded", "Failed", "Failed DB Names"])

            dfs: List[pd.DataFrame] = []
            header_written = False

            # Per-server stats
            total_dbs = 0
            succeeded = 0
            failed = 0
            failed_db_names: List[str] = []

            # Loop through databases for this server
            for _, row in group.iterrows():
                database = row["Database"]
                total_dbs += 1
                overall_total += 1  # overall attempt

                start_time = datetime.now()

                df, row_count, duration, status = fetch_data(
                    database=database,
                    server=server,
                    query=query,
                    timeout=DEFAULT_TIMEOUT_SECONDS,
                    encrypt=ENABLE_ODBC_ENCRYPTION
                )

                # Log per-database result
                log_sheet.append([
                    server,
                    database,
                    status,
                    row_count,
                    round(duration, 2),
                    start_time.strftime("%Y-%m-%d %H:%M:%S")
                ])

                # Tally
                if status == "Success":
                    succeeded += 1
                    overall_succeeded += 1
                    if df is not None and not df.empty:
                        if not header_written:
                            for r in dataframe_to_rows(df, index=False, header=True):
                                raw_data_sheet.append(r)
                            header_written = True
                        else:
                            for r in dataframe_to_rows(df, index=False, header=False):
                                raw_data_sheet.append(r)
                        dfs.append(df)
                else:
                    failed += 1
                    overall_failed += 1
                    failed_db_names.append(str(database))
                    overall_failed_list.append(f"{database}@{server}")

                # Small pause to reduce tempdb constraint races
                if DB_PAUSE_SECONDS > 0:
                    time.sleep(DB_PAUSE_SECONDS)

            # Add per-server summary row
            summary_sheet.append([
                server,
                total_dbs,
                succeeded,
                failed,
                ", ".join(failed_db_names) if failed_db_names else ""
            ])

            # Console summary for the server
            if failed > 0:
                print(f"⚠️ Server '{server}': {succeeded}/{total_dbs} databases succeeded.")
                print(f"   Failed: {', '.join(failed_db_names)}")
            else:
                print(f"✅ Server '{server}': All {total_dbs} databases succeeded.")

            # Save result for this server
            timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
            base_name = os.path.splitext(sql_file)[0]
            safe_server = server.replace("\\", "_").replace("/", "_").replace(":", "_")
            output_file = os.path.join(OUTPUT_DIR, f"{base_name}_{safe_server}_{timestamp}.xlsx")

            master_file.save(output_file)

            # Combined shape info (best-effort)
            combined_shape = (0, 0)
            if dfs:
                try:
                    combined_shape = pd.concat(dfs, ignore_index=True).shape
                except Exception:
                    combined_shape = (sum(len(d) for d in dfs), len(dfs[0].columns))

            print(f"✅ Saved result for Server {server}: {output_file}")
            print(f"📊 Combined shape for {server}: {combined_shape}\n")

        # --- Overall summary for this SQL file ---
        print(f"🧮 Overall for {sql_file}: {overall_succeeded}/{overall_total} databases succeeded; {overall_failed} failed")
        if overall_failed_list:
            print("   ❌ Failed DBs: " + ", ".join(overall_failed_list))
        print("")  # spacer

    beep_success()
    print("🎉 All done.")

if __name__ == "__main__":
    main()

📘 Loading DB targets from Excel (evaluated values)...

🔍 Running query from: CONT+PR $25 Variance_Live.txt

➡️ Processing Server: AHS-A2ATRN01.extapp.local
Connecting to DB='TranE120' on Server='AHS-A2ATRN01.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranE122' on Server='AHS-A2ATRN01.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranE129' on Server='AHS-A2ATRN01.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranE130' on Server='AHS-A2ATRN01.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranE132' on Server='AHS-A2ATRN01.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranE154' on Server='AHS-A2ATRN01.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranA279' on Server='AHS-A2ATRN01.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
✅ Server 'AHS-A2ATRN01.extapp.local': A

C:\Users\local_IN10052060\Temp\ipykernel_8128\679135434.py:436: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_shape = pd.concat(dfs, ignore_index=True).shape
C:\Users\local_IN10052060\Temp\ipykernel_8128\679135434.py:436: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_shape = pd.concat(dfs, ignore_index=True).shape


Connecting to DB='TranSDAC' on Server='ALPA2ATRN43.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranSMEC' on Server='ALPA2ATRN43.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranSAFC' on Server='ALPA2ATRN43.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranSROC' on Server='ALPA2ATRN43.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranSVSC' on Server='ALPA2ATRN43.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranSTCC' on Server='ALPA2ATRN43.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranSSMC' on Server='ALPA2ATRN43.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranSHAC' on Server='ALPA2ATRN43.extapp.local' using Driver='ODBC Driver 17 for SQL Server' ...
Connecting to DB='TranSCHC' on Server='ALPA2ATRN43.extapp.local' using Driver='ODBC Driv

In [5]:
import os
import pandas as pd

# --- CONFIGURATION ---
EXCEL_FOLDER = r"C:\Users\IN10052060\Downloads\CONT+PR $25 Variance_Live\file"
 # Update to your folder path
combined_df = pd.DataFrame()

# --- LOOP THROUGH EXCEL FILES ---
excel_files = [f for f in os.listdir(EXCEL_FOLDER) if f.endswith(".xlsx")]

print(f"Found {len(excel_files)} Excel files:")
for file in excel_files:
    file_path = os.path.join(EXCEL_FOLDER, file)
    try:
        # Read the 'RawData' sheet (adjust if needed)
        df = pd.read_excel(file_path, sheet_name="RawData")
        df["SourceFile"] = file  # Optional: track origin
        combined_df = pd.concat([combined_df, df], ignore_index=True)
        print(f"✅ Loaded: {file} ({df.shape[0]} rows)")
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")

# --- PREVIEW RESULT ---
print(f"\n📊 Combined shape: {combined_df.shape}")
print(combined_df.head())


Found 39 Excel files:
✅ Loaded: CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp.local_20260522-123423.xlsx (167 rows)
✅ Loaded: CONT+PR $25 Variance_Live_AHS-A2ATRN02.extapp.local_20260522-123542.xlsx (113 rows)
✅ Loaded: CONT+PR $25 Variance_Live_AHS-A2ATRN03.extapp.local_20260522-123649.xlsx (128 rows)
✅ Loaded: CONT+PR $25 Variance_Live_AHS-A2RSAS03.extapp.local_20260522-123807.xlsx (11 rows)
✅ Loaded: CONT+PR $25 Variance_Live_AHS-Care05.accretivehealth.local_20260522-123814.xlsx (0 rows)
✅ Loaded: CONT+PR $25 Variance_Live_AHS-TRAN01.accretivehealth.local_20260522-123928.xlsx (1 rows)
✅ Loaded: CONT+PR $25 Variance_Live_AHS-TRAN02.accretivehealth.local_20260522-123949.xlsx (5 rows)
✅ Loaded: CONT+PR $25 Variance_Live_AHS-TRAN08.accretivehealth.local_20260522-124250.xlsx (5 rows)
✅ Loaded: CONT+PR $25 Variance_Live_AHS-TRAN10.accretivehealth.local_20260522-130546.xlsx (3858 rows)
✅ Loaded: CONT+PR $25 Variance_Live_AHS-TRAN11.accretivehealth.local_20260522-130638.xlsx (19 rows)
✅ Loa

In [6]:
file = combined_df.copy()
file.shape

(13450, 44)

In [68]:
file.head()

,Key,Description_Servc,Billdate,actual_remit_date,Facilitycode,EncounterID,id,EncounterOpenBalance,EncounterTotalCharges,TotalOpenBalance,...,ActivityCode,LastAction,CreatedDateTime,ScheduleDateTime,Bill_Type,Description_Servc.1,Billdate.1,rn,actual_remit_date.1,SourceFile
0,E12011500023621,Day Surgery,2026-04-15,2026-04-28,E120,11500023621,21676,10295.96,18341.19,10295.96,...,System action taken to reset account age,60714.0,2026-05-18 16:11:35.263,2026-05-18,NaN,Day Surgery,2026-04-15,1,2026-04-28,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....
1,E12011500029380,Emergency,2026-05-27,2026-05-29,E120,11500029380,25233,496.26,15524.14,2395.96,...,NaN,NaN,NaT,NaT,NaN,Emergency,2026-05-27,1,2026-05-29,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....
2,E12212000003677,Day Surgery,2026-01-29,2026-04-13,E122,12000003677,1821,1778.71,13043.42,1778.71,...,System action taken to reset account age,60714.0,2026-06-06 14:23:35.907,2026-06-06,NaN,Day Surgery,2026-01-29,1,2026-04-13,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....
3,E12212000015286,Emergency,2026-04-16,2025-11-17,E122,12000015286,8264,1390.54,10003.75,4533.65,...,System action taken to reset account age,60714.0,2026-05-18 16:12:46.773,2026-05-18,NaN,Emergency,2026-04-16,1,2025-11-17,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....
4,E12212000020522,Outpatient,2026-04-16,2025-12-29,E122,12000020522,11543,4158.91,7617.20,8494.83,...,System action taken to reset account age,60714.0,2026-05-18 16:12:46.773,2026-05-18,NaN,Outpatient,2026-04-16,1,2025-12-29,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....


In [6]:
file['Facilitycode'] = file['Facilitycode'].str.strip()


In [7]:
Facility_code = pd.read_excel(r'C:\Users\IN10052060\OneDrive - R1\Lack Information\Latest Facility.xlsx', sheet_name = 'Sheet1')

In [8]:
Facility_code.shape

(297, 4)

In [9]:
Facility_code.drop_duplicates(subset = 'Facility code', inplace = True)

In [10]:
Facility_code.shape

(296, 4)

In [11]:
Facility_code['Facility Code'] = Facility_code['Facility code'].str.strip()

In [12]:
file.shape

(14368, 52)

In [13]:
Facility_code = Facility_code[['Facility code', 'Site in TB Tracker']]

In [14]:
Facility_code.head()

,Facility code,Site in TB Tracker
0,AAIL,Amita
1,HBIL,Amita
2,SAIL,Amita
3,BSTX,Austin
4,DCNT,Austin


In [15]:
facility_filter = file.merge(Facility_code, left_on = 'Facilitycode', right_on = 'Facility code', how = 'left')

In [16]:
facility_filter['Site'] = facility_filter['Site in TB Tracker']

In [17]:
facility_filter.head()

,Key,Description_Servc,Billdate,actual_remit_date,Facilitycode,EncounterID,id,EncounterOpenBalance,EncounterTotalCharges,TotalOpenBalance,...,ScheduleDateTime,Bill_Type,Description_Servc.1,Billdate.1,rn,actual_remit_date.1,SourceFile,Facility code,Site in TB Tracker,Site
0,E12011500023621,Day Surgery,2026-04-15,2026-04-28,E120,11500023621,21676,10295.96,18341.19,10295.96,...,2026-05-18,NaN,Day Surgery,2026-04-15,1,2026-04-28,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E120,Intermountain Health,Intermountain Health
1,E12011500029380,Emergency,2026-05-27,2026-05-29,E120,11500029380,25233,496.26,15524.14,2395.96,...,NaT,NaN,Emergency,2026-05-27,1,2026-05-29,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E120,Intermountain Health,Intermountain Health
2,E12212000003677,Day Surgery,2026-01-29,2026-04-13,E122,12000003677,1821,1778.71,13043.42,1778.71,...,2026-06-06,NaN,Day Surgery,2026-01-29,1,2026-04-13,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E122,Intermountain Health,Intermountain Health
3,E12212000015286,Emergency,2026-04-16,2025-11-17,E122,12000015286,8264,1390.54,10003.75,4533.65,...,2026-05-18,NaN,Emergency,2026-04-16,1,2025-11-17,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E122,Intermountain Health,Intermountain Health
4,E12212000020522,Outpatient,2026-04-16,2025-12-29,E122,12000020522,11543,4158.91,7617.20,8494.83,...,2026-05-18,NaN,Outpatient,2026-04-16,1,2025-12-29,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E122,Intermountain Health,Intermountain Health


In [18]:
facility_filter['Site_l'] = facility_filter['Site']

In [19]:
facility_filter['Site_l'] = facility_filter['Site_l'].str.strip().str.lower()

In [24]:
site_filter = pd.read_excel(r"C:\Users\IN10052060\Downloads\CONT+PR $25 Variance_Live\Filter.xlsx")
site_filter.shape

(32, 1)

In [25]:
site_filter['Sites'] = site_filter['Sites'].str.strip().str.lower()

In [26]:
site_filter.drop_duplicates(subset = 'Sites', inplace = True)

In [27]:
site_filter

,Sites
0,amita
1,austin
2,baltimore
3,birmingham
4,borgess
5,chmi
6,crittenton
7,detroit east
8,detroit west
9,flint


In [28]:
facility_filter = facility_filter[facility_filter['Site'].notna()]

In [29]:
facility_filter.head()

,Key,Description_Servc,Billdate,actual_remit_date,Facilitycode,EncounterID,id,EncounterOpenBalance,EncounterTotalCharges,TotalOpenBalance,...,Bill_Type,Description_Servc.1,Billdate.1,rn,actual_remit_date.1,SourceFile,Facility code,Site in TB Tracker,Site,Site_l
0,E12011500023621,Day Surgery,2026-04-15,2026-04-28,E120,11500023621,21676,10295.96,18341.19,10295.96,...,NaN,Day Surgery,2026-04-15,1,2026-04-28,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E120,Intermountain Health,Intermountain Health,intermountain health
1,E12011500029380,Emergency,2026-05-27,2026-05-29,E120,11500029380,25233,496.26,15524.14,2395.96,...,NaN,Emergency,2026-05-27,1,2026-05-29,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E120,Intermountain Health,Intermountain Health,intermountain health
2,E12212000003677,Day Surgery,2026-01-29,2026-04-13,E122,12000003677,1821,1778.71,13043.42,1778.71,...,NaN,Day Surgery,2026-01-29,1,2026-04-13,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E122,Intermountain Health,Intermountain Health,intermountain health
3,E12212000015286,Emergency,2026-04-16,2025-11-17,E122,12000015286,8264,1390.54,10003.75,4533.65,...,NaN,Emergency,2026-04-16,1,2025-11-17,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E122,Intermountain Health,Intermountain Health,intermountain health
4,E12212000020522,Outpatient,2026-04-16,2025-12-29,E122,12000020522,11543,4158.91,7617.20,8494.83,...,NaN,Outpatient,2026-04-16,1,2025-12-29,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E122,Intermountain Health,Intermountain Health,intermountain health


In [30]:
filter_site = facility_filter.merge(site_filter, left_on = 'Site_l', right_on = 'Sites', how = 'left')

In [31]:
filter_site = filter_site[filter_site['Sites'].notna()]

In [32]:
filter_site.shape

(4568, 57)

In [33]:
filter_site.head()

,Key,Description_Servc,Billdate,actual_remit_date,Facilitycode,EncounterID,id,EncounterOpenBalance,EncounterTotalCharges,TotalOpenBalance,...,Description_Servc.1,Billdate.1,rn,actual_remit_date.1,SourceFile,Facility code,Site in TB Tracker,Site,Site_l,Sites
0,E12011500023621,Day Surgery,2026-04-15,2026-04-28,E120,11500023621,21676,10295.96,18341.19,10295.96,...,Day Surgery,2026-04-15,1,2026-04-28,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E120,Intermountain Health,Intermountain Health,intermountain health,intermountain health
1,E12011500029380,Emergency,2026-05-27,2026-05-29,E120,11500029380,25233,496.26,15524.14,2395.96,...,Emergency,2026-05-27,1,2026-05-29,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E120,Intermountain Health,Intermountain Health,intermountain health,intermountain health
2,E12212000003677,Day Surgery,2026-01-29,2026-04-13,E122,12000003677,1821,1778.71,13043.42,1778.71,...,Day Surgery,2026-01-29,1,2026-04-13,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E122,Intermountain Health,Intermountain Health,intermountain health,intermountain health
3,E12212000015286,Emergency,2026-04-16,2025-11-17,E122,12000015286,8264,1390.54,10003.75,4533.65,...,Emergency,2026-04-16,1,2025-11-17,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E122,Intermountain Health,Intermountain Health,intermountain health,intermountain health
4,E12212000020522,Outpatient,2026-04-16,2025-12-29,E122,12000020522,11543,4158.91,7617.20,8494.83,...,Outpatient,2026-04-16,1,2025-12-29,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,E122,Intermountain Health,Intermountain Health,intermountain health,intermountain health


In [34]:
filter_site.drop(['Facility code', 'Site in TB Tracker', 'Site_l', 'Sites'], axis = 1, inplace = True)

In [35]:
filter_site.columns

Index(['Key', 'Description_Servc', 'Billdate', 'actual_remit_date',
       'Facilitycode', 'EncounterID', 'id', 'EncounterOpenBalance',
       'EncounterTotalCharges', 'TotalOpenBalance',
       'EncounterInsuranceBalance', 'EncounterPatientBalance',
       'CurrentPayorPlanCode', 'CurrentFinancialClass', 'ServiceFieldCode',
       'PatientType', 'DischargeDate', 'DFD', 'AdmitDate', 'CONT_Amt',
       'PRi_TotalClaimChargeAmount', 'Pri_TotalClaimPaymentAmount',
       'Remit_Date', 'PriPR', 'PostedPayment', 'Expected_Payment',
       'PatientPayment', 'TotalEncounterAdjustments', 'LatestAdjustmentDate',
       'LatestPostDate', 'IME_pay', 'PRIMARYPAYORPLANCODE',
       'SECONDARYPAYORPLANCODE', 'TertiaryPAYORPLANCODE',
       'CurrentPayerPlanName', 'CurrentMajorPayor', 'PrimaryPayerPlanName',
       'PrimaryMajorPayerName', 'SecondaryPayerPlanName',
       'SecondaryMajorPayerName', 'Contractual_on_remit', 'sr', 'ActivityCode',
       'LastAction', 'CreatedDateTime', 'ScheduleDateTime

In [36]:
facility_filter = filter_site.copy()

In [37]:
facility_filter.columns

Index(['Key', 'Description_Servc', 'Billdate', 'actual_remit_date',
       'Facilitycode', 'EncounterID', 'id', 'EncounterOpenBalance',
       'EncounterTotalCharges', 'TotalOpenBalance',
       'EncounterInsuranceBalance', 'EncounterPatientBalance',
       'CurrentPayorPlanCode', 'CurrentFinancialClass', 'ServiceFieldCode',
       'PatientType', 'DischargeDate', 'DFD', 'AdmitDate', 'CONT_Amt',
       'PRi_TotalClaimChargeAmount', 'Pri_TotalClaimPaymentAmount',
       'Remit_Date', 'PriPR', 'PostedPayment', 'Expected_Payment',
       'PatientPayment', 'TotalEncounterAdjustments', 'LatestAdjustmentDate',
       'LatestPostDate', 'IME_pay', 'PRIMARYPAYORPLANCODE',
       'SECONDARYPAYORPLANCODE', 'TertiaryPAYORPLANCODE',
       'CurrentPayerPlanName', 'CurrentMajorPayor', 'PrimaryPayerPlanName',
       'PrimaryMajorPayerName', 'SecondaryPayerPlanName',
       'SecondaryMajorPayerName', 'Contractual_on_remit', 'sr', 'ActivityCode',
       'LastAction', 'CreatedDateTime', 'ScheduleDateTime

In [38]:
PrimaryPPC = facility_filter['PRIMARYPAYORPLANCODE'].str.contains(
    r"\bself?\b", case=False, na=False
)

In [39]:
facility_filter = facility_filter[~PrimaryPPC]

In [58]:
amita = facility_filter['Site'] == 'Amita'
bill_date_gt_remit_date = facility_filter['Billdate'] > facility_filter['actual_remit_date']

In [67]:
facility_filter = facility_filter[~(amita & bill_date_gt_remit_date)]

dtype('<M8[ns]')

In [40]:
import time

start_time = time.time()

IMH_ATB = pd.read_excel(
    r"C:\Users\IN10052060\Downloads\IMH ATB.xlsb",
    sheet_name='HB ATB'
)

end_time = time.time()
elapsed_time = end_time - start_time

print(f"File loaded in {elapsed_time:.2f} seconds")

File loaded in 490.47 seconds


In [41]:
IMH_ATB.columns

Index(['PERIOD_ID', 'CLIENT', 'SITE_CD', 'FACILITY_TYPE', 'FACILITY_ID',
       'FACILITY_NAME', 'FACILITY_NPI', 'FACILITY_TAX_ID', 'ACCOUNT_NUMBER',
       'MRN',
       ...
       'Like Balances Flag', 'No Ins Pmt, Balance with Pt Flag',
       'Move to Secondary/Pt Flag', 'PR = Insurance Balance Flag',
       'AR>180, Account w/ No Pmt Flag', 'AR>180, Account w/ Pmt Flag',
       'CO 24 Flag', 'CO 167/234 Flag', 'Eligibility Denials',
       'Account in Appeal Flag'],
      dtype='object', length=137)

In [42]:
IMH_ATB = IMH_ATB.copy()

In [43]:
IMH_ATB = IMH_ATB[['SITE_CD', 'ACCOUNT_NUMBER', 'EXPECTED_REIMBURSEMENT']]

In [44]:
IMH_ATB['EXPECTED_REIMBURSEMENT'] = IMH_ATB['EXPECTED_REIMBURSEMENT'].fillna(0)

In [45]:
IMH_ATB['ACCOUNT_NUMBER'] = IMH_ATB['ACCOUNT_NUMBER'].astype(str).str.lstrip('0')

In [46]:
IMH_ATB['Key'] = IMH_ATB['SITE_CD'].astype(str) + IMH_ATB['ACCOUNT_NUMBER'].astype(str)

In [47]:
facility_filter['EncounterID'] = facility_filter['EncounterID'].astype(str).str.lstrip('0')

In [48]:
facility_filter['Unique'] = facility_filter['Facilitycode'].astype(str) + facility_filter['EncounterID'].astype(str)

In [49]:
facility_filter.head()

,Facilitycode,EncounterID,id,EncounterOpenBalance,EncounterTotalCharges,TotalOpenBalance,EncounterInsuranceBalance,EncounterPatientBalance,CurrentPayorPlanCode,CurrentFinancialClass,...,Contractual_on_remit,sr,ActivityCode,LastAction,CreatedDateTime,ScheduleDateTime,Bill_Type,SourceFile,Site,Unique
0,E120,11500021359,20960.0,8213.12,16234.67,8748.30,8213.12,0.0,1656,1.0,...,7270.27,1.0,System action taken to reset account age,60714.0,2026-05-18 16:11:35.263,2026-05-18,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12011500021359
1,E120,11500023621,21676.0,10295.96,18341.19,10295.96,10295.96,0.0,15,107.0,...,2100.33,1.0,System action taken to reset account age,60714.0,2026-05-18 16:11:35.263,2026-05-18,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12011500023621
2,E122,12000003677,1821.0,1778.71,13043.42,1778.71,1778.71,0.0,1579,1.0,...,NaN,1.0,System action taken to reset account age,60714.0,2026-05-18 16:12:46.773,2026-05-18,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000003677
3,E122,12000015286,8264.0,1390.54,10003.75,6259.63,1390.54,0.0,1882,3.0,...,NaN,1.0,System action taken to reset account age,60714.0,2026-05-18 16:12:46.773,2026-05-18,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000015286
4,E122,12000020522,11543.0,4158.91,7617.20,8270.37,4158.91,0.0,16,107.0,...,NaN,1.0,System action taken to reset account age,60714.0,2026-05-18 16:12:46.773,2026-05-18,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000020522


In [50]:
facility_filter.shape

(4183, 46)

In [51]:
facility_filter.drop_duplicates(subset = ['Unique'], inplace = True)

In [52]:
facility_IMH_ATB = facility_filter.copy()

In [53]:
facility_IMH_ATB.head()

,Facilitycode,EncounterID,id,EncounterOpenBalance,EncounterTotalCharges,TotalOpenBalance,EncounterInsuranceBalance,EncounterPatientBalance,CurrentPayorPlanCode,CurrentFinancialClass,...,Contractual_on_remit,sr,ActivityCode,LastAction,CreatedDateTime,ScheduleDateTime,Bill_Type,SourceFile,Site,Unique
0,E120,11500021359,20960.0,8213.12,16234.67,8748.30,8213.12,0.0,1656,1.0,...,7270.27,1.0,System action taken to reset account age,60714.0,2026-05-18 16:11:35.263,2026-05-18,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12011500021359
1,E120,11500023621,21676.0,10295.96,18341.19,10295.96,10295.96,0.0,15,107.0,...,2100.33,1.0,System action taken to reset account age,60714.0,2026-05-18 16:11:35.263,2026-05-18,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12011500023621
2,E122,12000003677,1821.0,1778.71,13043.42,1778.71,1778.71,0.0,1579,1.0,...,NaN,1.0,System action taken to reset account age,60714.0,2026-05-18 16:12:46.773,2026-05-18,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000003677
3,E122,12000015286,8264.0,1390.54,10003.75,6259.63,1390.54,0.0,1882,3.0,...,NaN,1.0,System action taken to reset account age,60714.0,2026-05-18 16:12:46.773,2026-05-18,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000015286
4,E122,12000020522,11543.0,4158.91,7617.20,8270.37,4158.91,0.0,16,107.0,...,NaN,1.0,System action taken to reset account age,60714.0,2026-05-18 16:12:46.773,2026-05-18,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000020522


In [54]:
Final_df = facility_IMH_ATB.merge(IMH_ATB, left_on = 'Unique', right_on = 'Key', how = 'left')

In [55]:
Final_df

,Facilitycode,EncounterID,id,EncounterOpenBalance,EncounterTotalCharges,TotalOpenBalance,EncounterInsuranceBalance,EncounterPatientBalance,CurrentPayorPlanCode,CurrentFinancialClass,...,CreatedDateTime,ScheduleDateTime,Bill_Type,SourceFile,Site,Unique,SITE_CD,ACCOUNT_NUMBER,EXPECTED_REIMBURSEMENT,Key
0,E120,11500021359,20960.0,8213.12,16234.67,8748.30,8213.12,0.00,1656,1.0,...,2026-05-18 16:11:35.263,2026-05-18 00:00:00,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12011500021359,E120,11500021359,15180.32,E12011500021359
1,E120,11500023621,21676.0,10295.96,18341.19,10295.96,10295.96,0.00,15,107.0,...,2026-05-18 16:11:35.263,2026-05-18 00:00:00,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12011500023621,E120,11500023621,17424.09,E12011500023621
2,E122,12000003677,1821.0,1778.71,13043.42,1778.71,1778.71,0.00,1579,1.0,...,2026-05-18 16:12:46.773,2026-05-18 00:00:00,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000003677,E122,12000003677,11857.59,E12212000003677
3,E122,12000015286,8264.0,1390.54,10003.75,6259.63,1390.54,0.00,1882,3.0,...,2026-05-18 16:12:46.773,2026-05-18 00:00:00,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000015286,E122,12000015286,3201.04,E12212000015286
4,E122,12000020522,11543.0,4158.91,7617.20,8270.37,4158.91,0.00,16,107.0,...,2026-05-18 16:12:46.773,2026-05-18 00:00:00,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000020522,E122,12000020522,6427.80,E12212000020522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4178,QMOR,9012420,1016982.0,3353.13,7452.77,3353.13,0.00,3353.13,SELF,SI,...,2026-05-12 11:21:45.027,2026-05-13 00:00:00,131,CONT+PR $25 Variance_Live_ALPTRAN36.accretiveh...,Quorum,QMOR9012420,NaN,NaN,NaN,NaN
4179,QFAR,3157026,368723.0,800.32,1565.71,1097.24,0.00,800.32,801002,2,...,2025-05-29 04:31:55.387,2025-06-05 00:00:00,131,CONT+PR $25 Variance_Live_ALPTRAN37.accretiveh...,Quorum,QFAR3157026,NaN,NaN,NaN,NaN
4180,QMUT,3347690,465638.0,560.43,17954.33,560.43,0.00,560.43,SELF,S,...,2026-02-16 09:02:06.917,2026-02-16 09:01:00,111,CONT+PR $25 Variance_Live_ALPTRAN37.accretiveh...,Quorum,QMUT3347690,NaN,NaN,NaN,NaN
4181,QMUT,3365055,485401.0,350.00,1842.96,49626.04,0.00,350.00,SELF,SI,...,2026-05-12 09:58:58.160,2026-05-19 00:00:00,131,CONT+PR $25 Variance_Live_ALPTRAN37.accretiveh...,Quorum,QMUT3365055,NaN,NaN,NaN,NaN


In [56]:
Final_df['PriPR']= Final_df['PriPR'].fillna(0)
Final_df['PostedPayment']= Final_df['PostedPayment'].fillna(0)


In [57]:
Final_df

,Facilitycode,EncounterID,id,EncounterOpenBalance,EncounterTotalCharges,TotalOpenBalance,EncounterInsuranceBalance,EncounterPatientBalance,CurrentPayorPlanCode,CurrentFinancialClass,...,CreatedDateTime,ScheduleDateTime,Bill_Type,SourceFile,Site,Unique,SITE_CD,ACCOUNT_NUMBER,EXPECTED_REIMBURSEMENT,Key
0,E120,11500021359,20960.0,8213.12,16234.67,8748.30,8213.12,0.00,1656,1.0,...,2026-05-18 16:11:35.263,2026-05-18 00:00:00,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12011500021359,E120,11500021359,15180.32,E12011500021359
1,E120,11500023621,21676.0,10295.96,18341.19,10295.96,10295.96,0.00,15,107.0,...,2026-05-18 16:11:35.263,2026-05-18 00:00:00,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12011500023621,E120,11500023621,17424.09,E12011500023621
2,E122,12000003677,1821.0,1778.71,13043.42,1778.71,1778.71,0.00,1579,1.0,...,2026-05-18 16:12:46.773,2026-05-18 00:00:00,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000003677,E122,12000003677,11857.59,E12212000003677
3,E122,12000015286,8264.0,1390.54,10003.75,6259.63,1390.54,0.00,1882,3.0,...,2026-05-18 16:12:46.773,2026-05-18 00:00:00,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000015286,E122,12000015286,3201.04,E12212000015286
4,E122,12000020522,11543.0,4158.91,7617.20,8270.37,4158.91,0.00,16,107.0,...,2026-05-18 16:12:46.773,2026-05-18 00:00:00,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000020522,E122,12000020522,6427.80,E12212000020522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4178,QMOR,9012420,1016982.0,3353.13,7452.77,3353.13,0.00,3353.13,SELF,SI,...,2026-05-12 11:21:45.027,2026-05-13 00:00:00,131,CONT+PR $25 Variance_Live_ALPTRAN36.accretiveh...,Quorum,QMOR9012420,NaN,NaN,NaN,NaN
4179,QFAR,3157026,368723.0,800.32,1565.71,1097.24,0.00,800.32,801002,2,...,2025-05-29 04:31:55.387,2025-06-05 00:00:00,131,CONT+PR $25 Variance_Live_ALPTRAN37.accretiveh...,Quorum,QFAR3157026,NaN,NaN,NaN,NaN
4180,QMUT,3347690,465638.0,560.43,17954.33,560.43,0.00,560.43,SELF,S,...,2026-02-16 09:02:06.917,2026-02-16 09:01:00,111,CONT+PR $25 Variance_Live_ALPTRAN37.accretiveh...,Quorum,QMUT3347690,NaN,NaN,NaN,NaN
4181,QMUT,3365055,485401.0,350.00,1842.96,49626.04,0.00,350.00,SELF,SI,...,2026-05-12 09:58:58.160,2026-05-19 00:00:00,131,CONT+PR $25 Variance_Live_ALPTRAN37.accretiveh...,Quorum,QMUT3365055,NaN,NaN,NaN,NaN


In [58]:
IH = Final_df['Site'] == 'Intermountain Health'
IH_start_E = Final_df['Facilitycode'].str.startswith('E')

In [59]:
IMH_Data = Final_df[IH & IH_start_E]

In [60]:
IMH_Data.shape

(405, 50)

In [61]:
IMH_Data = IMH_Data[(IMH_Data['EXPECTED_REIMBURSEMENT'] - (IMH_Data['PriPR'] + IMH_Data['PostedPayment'])) < 250]

In [62]:
IMH_Data.shape

(65, 50)

In [63]:
remaining_data = Final_df[~(IH & IH_start_E)]

In [64]:
Final_df.shape

(4183, 50)

In [65]:
IMH_Data.shape[0] + remaining_data.shape[0]

3843

In [66]:
IMH_Data.shape[0]

65

In [67]:
IMH_Data = IMH_Data[IMH_Data['Key'].notna()]

In [68]:
IMH_Data.shape[0]

65

In [69]:
remaining_data.shape[0]

3778

In [70]:
facility_filter = pd.concat([IMH_Data, remaining_data])

In [71]:
facility_filter.head()

,Facilitycode,EncounterID,id,EncounterOpenBalance,EncounterTotalCharges,TotalOpenBalance,EncounterInsuranceBalance,EncounterPatientBalance,CurrentPayorPlanCode,CurrentFinancialClass,...,CreatedDateTime,ScheduleDateTime,Bill_Type,SourceFile,Site,Unique,SITE_CD,ACCOUNT_NUMBER,EXPECTED_REIMBURSEMENT,Key
6,E122,12000036302,23184.0,4091.56,5953.94,4351.56,4091.56,0.0,2317,102.0,...,2026-05-20 11:05:12.880,2026-05-27,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000036302,E122,12000036302,1644.48,E12212000036302
8,E122,12000039199,25400.0,2811.72,4293.31,15317.69,2941.72,-130.0,2317,102.0,...,2026-05-20 11:05:13.537,2026-05-27,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000039199,E122,12000039199,1236.54,E12212000039199
47,E130,19500156977,128050.0,539.91,8025.64,539.91,539.91,0.0,1524,106.0,...,NaT,NaT,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E13019500156977,E130,19500156977,0.00,E13019500156977
62,E132,21500113020,84158.0,86843.83,135218.96,182526.61,86843.83,0.0,1975,105.0,...,2026-05-20 11:05:27.080,2026-05-27,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E13221500113020,E132,21500113020,0.00,E13221500113020
78,E132,21500195710,152150.0,448.57,455.26,19386.47,448.57,0.0,1980,105.0,...,2026-05-20 11:05:35.393,2026-05-27,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E13221500195710,E132,21500195710,178.37,E13221500195710


In [72]:
facility_filter.head()

,Facilitycode,EncounterID,id,EncounterOpenBalance,EncounterTotalCharges,TotalOpenBalance,EncounterInsuranceBalance,EncounterPatientBalance,CurrentPayorPlanCode,CurrentFinancialClass,...,CreatedDateTime,ScheduleDateTime,Bill_Type,SourceFile,Site,Unique,SITE_CD,ACCOUNT_NUMBER,EXPECTED_REIMBURSEMENT,Key
6,E122,12000036302,23184.0,4091.56,5953.94,4351.56,4091.56,0.0,2317,102.0,...,2026-05-20 11:05:12.880,2026-05-27,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000036302,E122,12000036302,1644.48,E12212000036302
8,E122,12000039199,25400.0,2811.72,4293.31,15317.69,2941.72,-130.0,2317,102.0,...,2026-05-20 11:05:13.537,2026-05-27,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E12212000039199,E122,12000039199,1236.54,E12212000039199
47,E130,19500156977,128050.0,539.91,8025.64,539.91,539.91,0.0,1524,106.0,...,NaT,NaT,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E13019500156977,E130,19500156977,0.00,E13019500156977
62,E132,21500113020,84158.0,86843.83,135218.96,182526.61,86843.83,0.0,1975,105.0,...,2026-05-20 11:05:27.080,2026-05-27,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E13221500113020,E132,21500113020,0.00,E13221500113020
78,E132,21500195710,152150.0,448.57,455.26,19386.47,448.57,0.0,1980,105.0,...,2026-05-20 11:05:35.393,2026-05-27,NaN,CONT+PR $25 Variance_Live_AHS-A2ATRN01.extapp....,Intermountain Health,E13221500195710,E132,21500195710,178.37,E13221500195710


In [73]:
facility_filter['Work Allocation'] = "Manual"
facility_filter['Write Off'] = ""
facility_filter['Repetitive'] = "Fresh"
facility_filter['Phase'] = ""
facility_filter['Drill Name'] = "CONT+PR > $25 Variance"
facility_filter['Invoicenumber'] = ""
facility_filter['StandardPatientType'] = ""
facility_filter['Admitdate'] = ""
facility_filter['Dischargedate'] = ""
facility_filter['t CODE'] = ""
facility_filter['major payer'] = ""
facility_filter['payer name'] = ""

In [74]:
facility_filter.rename({'ScheduleDateTime' : 'nextfollowupdate'}, axis = 1, inplace  =True)

In [75]:
facility_filter.columns

Index(['Facilitycode', 'EncounterID', 'id', 'EncounterOpenBalance',
       'EncounterTotalCharges', 'TotalOpenBalance',
       'EncounterInsuranceBalance', 'EncounterPatientBalance',
       'CurrentPayorPlanCode', 'CurrentFinancialClass', 'ServiceFieldCode',
       'PatientType', 'DischargeDate', 'DFD', 'AdmitDate', 'CONT_Amt',
       'PRi_TotalClaimChargeAmount', 'Pri_TotalClaimPaymentAmount',
       'Remit_Date', 'PriPR', 'PostedPayment', 'Expected_Payment',
       'PatientPayment', 'TotalEncounterAdjustments', 'LatestAdjustmentDate',
       'LatestPostDate', 'IME_pay', 'PRIMARYPAYORPLANCODE',
       'SECONDARYPAYORPLANCODE', 'TertiaryPAYORPLANCODE',
       'CurrentPayerPlanName', 'CurrentMajorPayor', 'PrimaryPayerPlanName',
       'PrimaryMajorPayerName', 'SecondaryPayerPlanName',
       'SecondaryMajorPayerName', 'Contractual_on_remit', 'sr', 'ActivityCode',
       'LastAction', 'CreatedDateTime', 'nextfollowupdate', 'Bill_Type',
       'SourceFile', 'Site', 'Unique', 'SITE_CD', 'A

In [76]:
col = ['Facilitycode', 'Site', 'EncounterID', 'Work Allocation', 'Write Off',
       'Repetitive', 'Phase', 'id', 'EncounterOpenBalance',
       'EncounterTotalCharges', 'TotalOpenBalance',
       'EncounterInsuranceBalance', 'EncounterPatientBalance',
       'CurrentPayorPlanCode', 'CurrentFinancialClass', 'ServiceFieldCode',
       'PatientType', 'DischargeDate', 'DFD', 'AdmitDate', 'CONT_Amt',
       'PRi_TotalClaimChargeAmount', 'Pri_TotalClaimPaymentAmount',
       'Remit_Date', 'PriPR', 'PostedPayment','Expected_Payment','Bill_Type','PatientPayment',
       'TotalEncounterAdjustments', 'LatestAdjustmentDate', 'LatestPostDate',
       'IME_pay', 'PRIMARYPAYORPLANCODE', 'SECONDARYPAYORPLANCODE',
       'TertiaryPAYORPLANCODE', 'CurrentPayerPlanName', 'CurrentMajorPayor',
       'PrimaryPayerPlanName', 'PrimaryMajorPayerName',
       'SecondaryPayerPlanName', 'SecondaryMajorPayerName',
       'Contractual_on_remit', 'sr', 'ActivityCode', 'LastAction',
       'CreatedDateTime', 'nextfollowupdate',
      'Drill Name', 'Invoicenumber',
       'StandardPatientType', 'Admitdate', 'Dischargedate', 't CODE',
       'major payer', 'payer name',
      'SITE_CD',
       'ACCOUNT_NUMBER', 'EXPECTED_REIMBURSEMENT']

In [77]:
facility_filter = facility_filter[col]

In [78]:
facility_filter.shape

(3843, 59)

In [79]:
medicare = facility_filter['PrimaryMajorPayerName'] == 'MEDICARE'
Pensacola = facility_filter['Site'] == 'Pensacola'
Jacksonville = facility_filter['Site'] == 'Jacksonville'

In [80]:
facility_filter = facility_filter[~(medicare & (Pensacola | Jacksonville))]

In [81]:
facility_filter['Site'].unique()

array(['Intermountain Health', 'Detroit East', 'Indianapolis', 'Flint',
       'Lifepoint', 'Pensacola', 'WI-Ministry', 'Milwaukee', 'Nashville',
       'Wichita', 'Tulsa', 'Detroit West', 'Jacksonville', 'Austin',
       'Waco', 'Crittenton', 'Swedish', 'Wheaton', 'Presence', 'Amita',
       'Quorum'], dtype=object)

In [82]:
Swedish = facility_filter['Site'] == 'Swedish'
Providence = facility_filter['Site'] == 'Providence'

In [83]:
facility_filter = facility_filter[~(Swedish | Providence)]

In [84]:
prime_meditech = facility_filter['Facilitycode'].isin(['JEIL', 'PJIL', 'PKIL', 'PMIL'])

In [85]:
facility_filter = facility_filter[~prime_meditech]

In [86]:
facility_filter['PrimaryMajorPayerName'].unique()

array(['MEDICAREHMO', 'COMMERCIAL/HMO', 'BLUE CROSS', 'MEDICAIDHMO',
       'MEDICARE', 'MEDICAID', 'SELF', 'AUTO/WC'], dtype=object)

In [87]:
MEDICAID = facility_filter["PrimaryMajorPayerName"].str.contains(
    r"\b(MEDICAID)\b", case=False, na=False, regex=True
)

BLUE_CROSS = facility_filter["PrimaryMajorPayerName"].str.contains(
    r"\b(BLUE CROSS)\b", case=False, na=False, regex=True
)

MEDICAIDHMO = facility_filter["PrimaryMajorPayerName"].str.contains(
    r"\b(MEDICAIDHMO)\b", case=False, na=False, regex=True
)

medicaid_bluecross = MEDICAID | BLUE_CROSS | MEDICAIDHMO

In [88]:
amita = facility_filter['Site'] == 'Amita'
Presence = facility_filter['Site'] == 'Presence'

In [89]:
facility_filter = facility_filter[~(medicaid_bluecross & (amita | Presence))]

In [93]:
austin = facility_filter['Site'] == 'Austin'

In [106]:
bill_112 = facility_filter['Bill_Type'] == 112

In [110]:
discharge_date_null = pd.isnull(facility_filter['DischargeDate'])

In [115]:
facility_filter = facility_filter[~(austin & (bill_112 | discharge_date_null)) ]

In [ ]:
amita = facility_filter['Site'] == 'Amita'

In [127]:
facility_filter['OB=EP-PP'] = facility_filter.apply(lambda x : 'Yes' if x['EncounterOpenBalance'] == x['Expected_Payment'] - x['PostedPayment'] else 'No', axis =1)

In [129]:
yes = facility_filter['OB=EP-PP'] == 'Yes'


In [124]:
facility_filter = facility_filter.copy()

In [132]:
facility_filter = facility_filter[~(amita & yes)]

In [134]:
IMH = facility_filter['Site'] == 'Intermountain Health'

In [136]:
facility_filter['RC = TC'] = facility_filter.apply( lambda x : 'Yes' if x['PRi_TotalClaimChargeAmount'] == x['EncounterTotalCharges'] else 'No', axis =1)

In [138]:
No = facility_filter['RC = TC'] == 'No'

In [143]:
facility_filter['RC = TC'].value_counts()

RC = TC
No    418
Name: count, dtype: int64

In [144]:
facility_filter

,Facilitycode,Site,EncounterID,Work Allocation,Write Off,Repetitive,Phase,id,EncounterOpenBalance,EncounterTotalCharges,...,Admitdate,Dischargedate,t CODE,major payer,payer name,SITE_CD,ACCOUNT_NUMBER,EXPECTED_REIMBURSEMENT,OB=EP-PP,RC = TC
6,E122,Intermountain Health,12000036302,Manual,,Fresh,,23184.0,4091.56,5953.94,...,,,,,,E122,12000036302,1644.48,No,No
8,E122,Intermountain Health,12000039199,Manual,,Fresh,,25400.0,2811.72,4293.31,...,,,,,,E122,12000039199,1236.54,No,No
47,E130,Intermountain Health,19500156977,Manual,,Fresh,,128050.0,539.91,8025.64,...,,,,,,E130,19500156977,0.00,No,No
62,E132,Intermountain Health,21500113020,Manual,,Fresh,,84158.0,86843.83,135218.96,...,,,,,,E132,21500113020,0.00,No,No
78,E132,Intermountain Health,21500195710,Manual,,Fresh,,152150.0,448.57,455.26,...,,,,,,E132,21500195710,178.37,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4178,QMOR,Quorum,9012420,Manual,,Fresh,,1016982.0,3353.13,7452.77,...,,,,,,NaN,NaN,NaN,No,No
4179,QFAR,Quorum,3157026,Manual,,Fresh,,368723.0,800.32,1565.71,...,,,,,,NaN,NaN,NaN,No,No
4180,QMUT,Quorum,3347690,Manual,,Fresh,,465638.0,560.43,17954.33,...,,,,,,NaN,NaN,NaN,No,No
4181,QMUT,Quorum,3365055,Manual,,Fresh,,485401.0,350.00,1842.96,...,,,,,,NaN,NaN,NaN,No,No


In [145]:
facility_filter['Bill_Type'].unique()

array([nan, '                              ',
       '137                           ', '131                           ',
       '111                           ', 111,
       '117                           ', 851.0, 131.0,
       '141                           ', '110                           ',
       857.0, 137.0, 117.0, '121                           ', 132.0,
       110.0, 231.0, 181, 133.0, 121.0, 221.0], dtype=object)

In [146]:
import os
import time
import pandas as pd
from openpyxl import Workbook, load_workbook

start_time = time.time()

file_path = r"C:\Users\IN10052060\OneDrive - R1\Missing Drills\Contractual\CON+PR gt $25 Variance 2026_05_29.xlsx"

# Step 1: Check if file exists, delete if yes
if os.path.exists(file_path):
    os.remove(file_path)
    print("🗑️ Existing file deleted.")

# Step 2: Create a fresh Excel file
wb = Workbook()
wb.save(file_path)
print("📂 New Excel file created.")

# Step 3: Export your DataFrame to the new file
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a') as writer:
    facility_filter.to_excel(writer, sheet_name='Transform Data', index=False)

# Step 4: Export your DataFrame to the new file
with pd.ExcelWriter(file_path, engine='openpyxl', mode='a') as writer:
    file.to_excel(writer, sheet_name='Full Data', index=False)

end_time = time.time()
elapsed_time = end_time - start_time

print(f"✅ Data successfully written to the sheet in {elapsed_time:.2f} seconds")


📂 New Excel file created.
✅ Data successfully written to the sheet in 15.10 seconds
